In [1]:
# 00_build_graph.ipynb
# =============================================================================
# Build the heterogeneous SMEsD enterprise graph used by all downstream
# notebooks, with a leakage audit and DUAL label masks on one shared graph.
#
# Non-financial setting: company node features are 3 basic attributes plus a
# lawsuit-aggregate block (no financial ratios). Persons are metapath2vec
# embeddings. 20 relation types connect company/person nodes.
#
# Leakage control: a small number of companies recur across the time-based
# train/val/test splits under the same global id. We therefore build:
#   Mask A (strict, primary)   : each duplicated firm kept only in its LATEST
#                                split, so train/val/test company sets are disjoint.
#   Mask B (as-shipped, reference) : original split assignment kept unchanged.
# Nodes/edges/features are identical across masks; only the label masks differ.
# =============================================================================
import pickle
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch_geometric.data import HeteroData

# --- Paths (never printed, for anonymity) ------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
DATA_DIR = ROOT / "data" / "external" / "ComRisk" / "data"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.exists(), "SMEsD ComRisk data directory not found."


def load(name):
    with open(DATA_DIR / name, "rb") as f:
        return pickle.load(f)


# --- 1. Auxiliary files ------------------------------------------------------
node2index = load("node2index.pkl")
split_idx = load("split_data_idx.pkl")
meta_emb = load("meta_emb.pkl")

_k = next(iter(node2index))
name2idx = ({str(k): int(v) for k, v in node2index.items()} if isinstance(_k, str)
            else {str(v): int(k) for k, v in node2index.items()})
idx2name = {v: k for k, v in name2idx.items()}

company_emb = np.asarray(meta_emb[0], dtype=np.float32)
person_emb = np.asarray(meta_emb[1], dtype=np.float32)
N_COMPANY, N_PERSON = company_emb.shape[0], person_emb.shape[0]
all_ids = np.fromiter(name2idx.values(), dtype=np.int64)
GLOBAL_N = int(all_ids.max()) + 1
print(f"nodes: companies={N_COMPANY:,} persons={N_PERSON:,} "
      f"global_slots={GLOBAL_N:,} (gaps={GLOBAL_N - len(all_ids)})")

# --- 2. Load splits, confirm length-5 + row alignment ------------------------
FIELDS = ["risk_data", "company_attr", "hete_graph", "hyp_graph", "label"]
splits = {}
for nm, fn in [("train", "train_data.pkl"),
               ("valid", "validate_data.pkl"),
               ("test", "test_data.pkl")]:
    obj = load(fn)
    assert isinstance(obj, (list, tuple)) and len(obj) == 5
    splits[nm] = dict(zip(FIELDS, obj))
    ca = np.asarray(splits[nm]["company_attr"])
    lab = np.asarray(splits[nm]["label"])
    assert ca.shape[0] == lab.shape[0] and ca.shape[1] == 3
    print(f"{nm:5s} (as-shipped): companies={ca.shape[0]:,} "
          f"pos={int(lab.sum()):,} ({lab.mean():.2%})")

split_global_ids = {nm: np.asarray(ids, dtype=np.int64)
                    for nm, ids in zip(["train", "valid", "test"], split_idx)}

# --- 3. Leakage audit: quantify cross-split duplication -----------------------
sets = {nm: set(split_global_ids[nm].tolist()) for nm in ["train", "valid", "test"]}
dup_tv = sets["train"] & sets["valid"]
dup_tt = sets["train"] & sets["test"]
dup_vt = sets["valid"] & sets["test"]
all_dups = dup_tv | dup_tt | dup_vt
print(f"\nLEAKAGE AUDIT: duplicated companies across splits = {len(all_dups)} "
      f"(train-valid={len(dup_tv)} train-test={len(dup_tt)} valid-test={len(dup_vt)})")
print(f"union of labelled companies = {len(sets['train'] | sets['valid'] | sets['test']):,}")

# --- 4. Lawsuit encoding cardinalities ---------------------------------------
cause, court, category = set(), set(), set()
for s in splits.values():
    for events in s["risk_data"].values():
        for ev in events:
            cause.add(int(ev[0])); court.add(int(ev[1])); category.add(int(ev[2]))
N_CAUSE, N_COURT, N_CAT = max(cause) + 1, max(court) + 1, max(category) + 1
RISK_W = N_CAUSE + N_COURT + N_CAT + 1
BASIC_W = 3
print(f"lawsuit codes: cause={N_CAUSE} court={N_COURT} category={N_CAT} "
      f"-> lawsuit-aggregate width={RISK_W}; company feature dim={BASIC_W + RISK_W}")

# --- 5. Global company feature matrix ----------------------------------------
company_x = np.zeros((GLOBAL_N, BASIC_W + RISK_W), dtype=np.float32)
company_mask = np.zeros(GLOBAL_N, dtype=bool)
for nm in ["train", "valid", "test"]:
    ca = np.asarray(splits[nm]["company_attr"], dtype=np.float32)
    gids = split_global_ids[nm]
    company_x[gids, :BASIC_W] = ca
    company_mask[gids] = True
    for gid, events in splits[nm]["risk_data"].items():
        gid = int(gid)
        cause_c = np.zeros(N_CAUSE, np.float32); court_c = np.zeros(N_COURT, np.float32)
        cat_c = np.zeros(N_CAT, np.float32); times = []
        for ev in events:
            cause_c[int(ev[0])] += 1; court_c[int(ev[1])] += 1
            cat_c[int(ev[2])] += 1; times.append(float(ev[3]))
        company_x[gid, BASIC_W:] = np.concatenate(
            [cause_c, court_c, cat_c, [np.mean(times) if times else 0.0]])

# --- 6. Labels + dual masks (A strict, B as-shipped) -------------------------
y = np.full(GLOBAL_N, -1, dtype=np.int64)
for nm in ["train", "valid", "test"]:
    y[split_global_ids[nm]] = np.asarray(splits[nm]["label"], dtype=np.int64)


def empty_masks():
    return {k: np.zeros(GLOBAL_N, dtype=bool) for k in ["train", "valid", "test"]}


mask_B = empty_masks()
for nm in ["train", "valid", "test"]:
    mask_B[nm][split_global_ids[nm]] = True

mask_A = empty_masks()
assigned = np.zeros(GLOBAL_N, dtype=bool)
for nm in ["test", "valid", "train"]:          # latest first
    gids = split_global_ids[nm]
    take = gids[~assigned[gids]]
    mask_A[nm][take] = True
    assigned[take] = True

for tag, M in [("A (strict)", mask_A), ("B (as-shipped)", mask_B)]:
    ov = int((M["train"] & M["valid"]).sum() + (M["train"] & M["test"]).sum()
             + (M["valid"] & M["test"]).sum())
    n = {k: int(M[k].sum()) for k in M}
    pos = {k: int(y[M[k]].sum()) for k in M}
    print(f"mask {tag:14s}: train={n['train']:,}({pos['train']} pos) "
          f"val={n['valid']:,}({pos['valid']}) test={n['test']:,}({pos['test']}) "
          f"overlap={ov}" + ("  disjoint" if ov == 0 else "  overlap kept"))
assert int((mask_A["train"] & mask_A["valid"]).sum()
           + (mask_A["train"] & mask_A["test"]).sum()
           + (mask_A["valid"] & mask_A["test"]).sum()) == 0

# --- 7. Parse + merge heterogeneous edges ------------------------------------
def parse_hete(hg):
    ei, et, ew = (hg[0], hg[1], hg[2]) if isinstance(hg, (list, tuple)) \
        else (hg["edge_index"], hg["edge_type"], hg["edge_weight"])
    ei = np.asarray(ei)
    if ei.shape[0] != 2:
        ei = ei.T
    return ei.astype(np.int64), np.asarray(et).astype(np.int64), np.asarray(ew, np.float32)


edges = {}
for nm in ["train", "valid", "test"]:
    ei, et, ew = parse_hete(splits[nm]["hete_graph"])
    for k in range(ei.shape[1]):
        edges[(int(ei[0, k]), int(ei[1, k]))] = (int(et[k]), float(ew[k]))
src = np.array([s for s, _ in edges], np.int64)
dst = np.array([d for _, d in edges], np.int64)
etype = np.array([v[0] for v in edges.values()], np.int64)
eweight = np.array([v[1] for v in edges.values()], np.float32)
rel_ids = sorted(set(etype.tolist()))
print(f"\nmerged edges: {len(edges):,}  relations={rel_ids}")

# --- 8. Assemble PyG HeteroData ----------------------------------------------
comp_global = np.where(company_mask)[0]
comp_g2l = {int(g): i for i, g in enumerate(comp_global)}
person_global = np.arange(N_COMPANY, N_COMPANY + N_PERSON, dtype=np.int64)
pers_g2l = {int(g): i for i, g in enumerate(person_global)}


def sel(mask_dict, split, gids):
    return torch.tensor(mask_dict[split][gids])


data = HeteroData()
data["company"].x = torch.tensor(company_x[comp_global], dtype=torch.float32)
data["company"].y = torch.tensor(y[comp_global], dtype=torch.long)
data["company"].train_mask = sel(mask_A, "train", comp_global)
data["company"].val_mask = sel(mask_A, "valid", comp_global)
data["company"].test_mask = sel(mask_A, "test", comp_global)
data["company"].train_mask_repro = sel(mask_B, "train", comp_global)
data["company"].val_mask_repro = sel(mask_B, "valid", comp_global)
data["company"].test_mask_repro = sel(mask_B, "test", comp_global)
data["company"].num_nodes = len(comp_global)
assert person_emb.shape[0] == len(person_global)
data["person"].x = torch.tensor(person_emb, dtype=torch.float32)
data["person"].num_nodes = len(person_global)


def type_local(gid):
    return ("company", comp_g2l[gid]) if gid in comp_g2l else ("person", pers_g2l[gid])


buckets = {}
for s, d, r, w in zip(src, dst, etype, eweight):
    st, sl = type_local(int(s)); dt, dl = type_local(int(d))
    buckets.setdefault((st, f"rel_{r}", dt), [[], [], []])
    buckets[(st, f"rel_{r}", dt)][0].append(sl)
    buckets[(st, f"rel_{r}", dt)][1].append(dl)
    buckets[(st, f"rel_{r}", dt)][2].append(w)
for (st, rel, dt), (s_l, d_l, w_l) in buckets.items():
    store = data[st, rel, dt]
    store.edge_index = torch.tensor([s_l, d_l], dtype=torch.long)
    store.edge_weight = torch.tensor(w_l, dtype=torch.float32)
print(data)

# --- 9. Persist for downstream notebooks -------------------------------------
torch.save(data, PROCESSED / "smesd_hetero.pt")
torch.save({"comp_global": comp_global, "person_global": np.array(person_global),
            "idx2name": idx2name, "rel_ids": rel_ids,
            "dup_ids": np.array(sorted(all_dups), dtype=np.int64)},
           PROCESSED / "smesd_idmaps.pt")
print("\n00_build_graph complete: shared graph saved with masks A (strict) and "
      "B (as-shipped).")


/usr/local/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nodes: companies=3,976 persons=2,405 global_slots=6,381 (gaps=11)
train (as-shipped): companies=2,816 pos=1,621 (57.56%)
valid (as-shipped): companies=721 pos=354 (49.10%)
test  (as-shipped): companies=491 pos=318 (64.77%)

LEAKAGE AUDIT: duplicated companies across splits = 51 (train-valid=35 train-test=17 valid-test=1)
union of labelled companies = 3,976
lawsuit codes: cause=11 court=4 category=4 -> lawsuit-aggregate width=20; company feature dim=23
mask A (strict)    : train=2,765(1590 pos) val=720(354) test=491(318) overlap=0  disjoint
mask B (as-shipped): train=2,816(1621 pos) val=721(354) test=491(318) overlap=53  overlap kept



merged edges: 13,336  relations=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
HeteroData(
  company={
    x=[3976, 23],
    y=[3976],
    train_mask=[3976],
    val_mask=[3976],
    test_mask=[3976],
    train_mask_repro=[3976],
    val_mask_repro=[3976],
    test_mask_repro=[3976],
    num_nodes=3976,
  },
  person={
    x=[2405, 32],
    num_nodes=2405,
  },
  (company, rel_0, person)={
    edge_index=[2, 698],
    edge_weight=[698],
  },
  (person, rel_1, company)={
    edge_index=[2, 698],
    edge_weight=[698],
  },
  (company, rel_2, person)={
    edge_index=[2, 883],
    edge_weight=[883],
  },
  (person, rel_3, company)={
    edge_index=[2, 883],
    edge_weight=[883],
  },
  (company, rel_6, company)={
    edge_index=[2, 938],
    edge_weight=[938],
  },
  (company, rel_7, company)={
    edge_index=[2, 942],
    edge_weight=[942],
  },
  (company, rel_9, person)={
    edge_index=[2, 3657],
    edge_weight=[3657],
  },
  (person, rel_8, company)={
    